# SmartReads: Genre‑Specific Recommender
Content‑based filtering with **TF‑IDF + Cosine** over `bookDesc + bookGenres`,
then ranking by a weighted mix of **similarity**, **popularity** (ratingCount + reviewCount), and **quality** (bookRating).


In [ ]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
df = pd.read_csv('data/raw/goodreads_genre_clean_5k.csv', dtype=str)
df.head(3)

## Filter by genre

In [ ]:
GENRE_COL = 'bookGenres'
DESC_COL = 'bookDesc'
TITLE_COL = 'bookTitle'
df_sci = df[df[GENRE_COL].astype(str).str.lower().str.contains('science fiction', na=False)]
len(df_sci)

## Build TF‑IDF (genres emphasized)

In [ ]:
genres = df_sci[GENRE_COL].fillna('').astype(str).str.replace('[\[\]\'\"]','', regex=True).str.replace(',', ' ', regex=False)
desc = df_sci[DESC_COL].fillna('').astype(str)
combined = (genres + ' ' + genres + ' ' + desc).values
tfidf = TfidfVectorizer(stop_words='english', max_features=50000, ngram_range=(1,2))
X = tfidf.fit_transform(combined)
X.shape

## Rank by similarity + popularity + quality

In [ ]:
def rank(df_filt, X, query_text=None, topk=10, weights=(0.6,0.25,0.15)):
    from sklearn.feature_extraction.text import TfidfVectorizer
    if query_text:
        tfidf_local = TfidfVectorizer(stop_words='english', max_features=50000, ngram_range=(1,2))
        X_local = tfidf_local.fit_transform(combined)
        qv = tfidf_local.transform([query_text])
        sim = cosine_similarity(qv, X_local).ravel()
    else:
        sim = np.zeros(len(df_filt))
    RC_COL = 'ratingCount'
    REV_COL = 'reviewCount'
    RATING_COL = 'bookRating'
    pop = df_filt[[c for c in [RC_COL, REV_COL] if c in df_filt.columns]].apply(pd.to_numeric, errors='coerce').fillna(0.0).sum(axis=1).values.reshape(-1,1)
    pop = MinMaxScaler().fit_transform(pop).ravel() if len(pop) else np.zeros(len(df_filt))
    qual = pd.to_numeric(df_filt[RATING_COL], errors='coerce').fillna(0.0).values.reshape(-1,1)
    qual = MinMaxScaler().fit_transform(qual).ravel()
    w_sim, w_pop, w_qual = weights
    score = w_sim*sim + w_pop*pop + w_qual*qual
    order = np.argsort(-score)[:topk]
    cols = [c for c in ['bookTitle','bookAuthors', GENRE_COL, RATING_COL, RC_COL, REV_COL] if c in df_filt.columns]
    out = df_filt.iloc[order][cols].copy()
    out['score'] = score[order]
    return out

rank(df_sci, X, query_text='space opera', topk=5)